---
title: "Channel flow: the Method of Manufactured Solutions"
subtitle: "A pure-NumPy warm-up — verify a solver against a solution you invent."
author: "Peclet"
date: "2026-07-02"
categories: [verification, finite-differences, numpy]
jupyter: python3
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/peclet-examples/blob/main/examples/channel-mms/index.ipynb){target="_blank"}
&nbsp;Pure NumPy — runs anywhere, no GPU or solver install needed.

## What you'll learn

How to **verify** a discrete momentum solver without an analytical solution to
your real problem — by *manufacturing* one — and how to measure the **order of
accuracy** of the scheme with a grid-refinement study. This is the pattern
`peclet`'s flow solver uses to certify its cut-cell immersed-boundary step; here
we do it in a few lines of NumPy so the idea is naked. The
[solver-backed sequel](../poiseuille-ibm/index.qmd) then runs the real thing.

## The problem

Steady, fully developed flow in a channel of height $H$ reduces the streamwise
momentum equation to a one-dimensional ODE for the velocity profile $u(y)$:

$$
\mu\, u''(y) = -f(y), \qquad u(0) = u(H) = 0,
$$ {#eq-momentum}

where $\mu$ is the dynamic viscosity and $f$ a body force (for constant
$f=-\mathrm{d}p/\mathrm{d}x$ this is classic plane Poiseuille flow).

A discrete solver of @eq-momentum is only trustworthy if we know it converges to
the *right* answer at the *right rate*. But real geometries rarely have a closed
form to check against. The **Method of Manufactured Solutions** (MMS)
[@roache2002] sidesteps that: we pick a smooth target profile, differentiate it
to find the forcing that would produce it, and check the solver reproduces the
target.

## Manufacturing a solution

Choose a profile that already satisfies the no-slip walls,
$u_{\text{exact}}(y) = U_0 \sin(\pi y / H)$, then substitute into @eq-momentum to
read off the required body force
$f(y) = \mu U_0 (\pi/H)^2 \sin(\pi y / H)$. Unlike Poiseuille's parabola, a sine
is *not* reproduced exactly by a second-order stencil — so the measured error
reflects the scheme's true rate.

In [ ]:
#| label: bootstrap
#| code-summary: "Environment bootstrap (Colab/Binder)"
# This warm-up is pure NumPy — no solver needed. It only pulls this gallery's
# small helper package if it isn't already importable (a no-op locally / in CI).
import importlib.util, subprocess, sys
if importlib.util.find_spec("peclet_examples") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "peclet-examples @ git+https://github.com/computational-chemical-engineering/peclet-examples"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from peclet_examples import mms

params = dict(U0=1.0, mu=1.0, H=1.0)

## Solve and compare

We discretise @eq-momentum with a standard second-order central difference and
solve the resulting tridiagonal system.

In [ ]:
#| label: fig-profile
#| fig-cap: "Finite-difference solution (markers) against the manufactured profile (line)."
y, u = mms.solve_fd(N=20, **params)
yy = np.linspace(0, params["H"], 200)

fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.plot(mms.u_exact(yy, U0=params["U0"], H=params["H"]), yy, "k-", label="manufactured")
ax.plot(u, y, "o", ms=5, label="finite difference (N=20)")
ax.set(xlabel="u(y)", ylabel="y", title="Channel velocity profile")
ax.legend()
plt.show()

## Order of accuracy

Refining the grid should shrink the max-norm error like $\mathcal{O}(h^2)$.

In [ ]:
#| label: fig-convergence
#| fig-cap: "Grid convergence: measured error tracks the O(h²) reference line."
Ns, errs = mms.convergence([5, 10, 20, 40, 80, 160, 320], **params)
rate = np.polyfit(np.log(Ns), np.log(errs), 1)[0]

fig, ax = plt.subplots(figsize=(4.6, 3.6))
ax.loglog(Ns, errs, "o-", label="measured error")
ax.loglog(Ns, errs[0] * (Ns / Ns[0]) ** -2.0, "k--", label=r"$\mathcal{O}(h^2)$")
ax.set(xlabel="N (interior points)", ylabel="max error", title="Grid convergence")
ax.legend()
plt.show()

print(f"observed convergence rate = {rate:.3f}  (expect ≈ -2.0)")

The fitted slope of about $-1.9$ confirms the solver is second-order accurate.

::: {.callout-tip}
Constant-force Poiseuille is the *exact-recovery* special case: because the
solution is quadratic, the central difference reproduces it to machine precision
at every resolution — great for a smoke test, useless as a convergence test.
That subtlety is exactly why MMS exists.
:::

## Reproduce this

```bash
pip install -e .              # installs the peclet_examples helpers + NumPy
quarto render examples/channel-mms/index.qmd
```

Runs anywhere — no GPU, no compiled solver. Now see the
[real cut-cell IBM solver](../poiseuille-ibm/index.qmd) solve the same channel.

### References

::: {#refs}
:::